# Qwen3 与 Aya 结果对比

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

RUN_ROOT = Path("/content/drive/MyDrive/sft_v4/model_runs")
MODELS = ["qwen3_8b", "aya_expanse_8b"]
configs = [json.loads((RUN_ROOT / name / "run_config.json").read_text(encoding="utf-8")) for name in MODELS]

# 数据不同仍然停止；训练参数不同则明确列出，允许查看已有结果。
assert configs[0]["data_sha256"] == configs[1]["data_sha256"], "两次运行的数据内容或划分不一致，不能直接对比"
keys = ["epochs", "max_length", "seed", "effective_batch_size", "learning_rate", "optimizer_steps"]
settings = pd.DataFrame({name: {key: config.get(key) for key in keys} for name, config in zip(MODELS, configs)})
settings["一致"] = settings[MODELS[0]].eq(settings[MODELS[1]])
display(settings)
if not settings["一致"].all():
    print("注意：训练配置存在差异。以下展示各次运行的实际结果，不属于完全相同训练配置的对照实验。")

rows = []
for name in MODELS:
    metrics = json.loads((RUN_ROOT / name / "metrics.json").read_text(encoding="utf-8"))
    scores = metrics["by_language"]  # 官方各语言独立排名；不把 ALL 当作排名分数
    for language, values in scores.items():
        rows.append({"model": name, "language": language, **values})
display(pd.DataFrame(rows).set_index(["model", "language"]))
